# V-JEPA 2 ViT-L trên EK100: eval trên **video gốc** (tải về, eval, xóa ngay)

**Vì sao cần notebook này:** dataset `download-ek100` là bản **đã encode lại** (`scale=-2:256`, `crf 23`).
Pipeline của V-JEPA 2 resize cạnh ngắn về **292** rồi mới crop 256, nên với video 256p ảnh bị phóng to lên (mờ) và có
nhiễu nén. Kết quả là điểm bị kéo xuống: 28.6 trên bản encode lại, so với khoảng 31 khi chạy trên video gốc.
Video gốc (1920x1080, tổng khoảng **184 GB**) lại không vừa một Kaggle dataset, nên notebook này làm như sau:
1. Tải song song (aria2c) một vài video gốc từ `data.bris.ac.uk` vào `/kaggle/tmp`, kiểm tra từng file với
   `EPIC_100_video_info.csv` (độ phân giải, fps, thời lượng).
2. Eval ngay trên 2 GPU (model chỉ load 1 lần), tính **cả 2 giao thức** (clip neo theo code gốc `official`
   và giao thức chuẩn EK100 `action_start`) trong cùng một lần tải.
3. Xóa video, rồi tải video tiếp theo.

**Cài đặt:** GPU **T4 x2**, Internet **On**. Không cần add dataset video.
Phần chậm nhất là tải video (tùy băng thông, khoảng 3–8 giờ). Nếu hết 12 giờ mà chưa xong:
*Save Version*, rồi ở lần chạy sau *Add Input* output của version này. Notebook tự tiếp tục từ chỗ dừng.

In [ ]:
BRANCH = "claude/vjepa2-verify-improve-dsfgtg"
VJEPA2_COMMIT = "204698b45b3712590f06245fbfba32d3be539812"  # version the code was checked against
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!git clone -q https://github.com/facebookresearch/vjepa2.git /kaggle/working/vjepa2 && cd /kaggle/working/vjepa2 && git checkout -q $VJEPA2_COMMIT
!pip install -q decord webdataset timm einops
import os
os.environ["VJEPA2_ROOT"] = "/kaggle/working/vjepa2"
R = "/kaggle/working/repo"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq install -y aria2 > /dev/null 2>&1 && aria2c --version | head -1

In [ ]:
# Checkpoints (~1.3GB encoder+predictor, probe) -> /tmp so they do not count toward the 20GB output limit
!mkdir -p /tmp/ckpt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/vitl.pt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/evals/ek100-vitl-256.pt
!ls -lh /tmp/ckpt

In [ ]:
# Official annotations (the full train csv is required: probe output classes are derived from it)
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /kaggle/working/ek100-ann
!ls /kaggle/input; find /kaggle/input -maxdepth 3 | head -30

In [ ]:
# Resume: bring back logits of an earlier run of THIS notebook (folder logits_orig in an attached output)
import glob, os, shutil
OUT = "/kaggle/working/logits_orig"
prev = glob.glob("/kaggle/input/**/logits_orig", recursive=True)
if prev:
    shutil.copytree(prev[0], OUT, dirs_exist_ok=True)
    print("resumed from", prev[0], "->", sorted(os.listdir(OUT)))
else:
    print("fresh run")

In [ ]:
# Clip index: videos will be downloaded to /kaggle/tmp/ek100/<P>/videos/<id>.MP4
!python $R/scripts/prepare_ek100.py --download_layout --video_root /kaggle/tmp/ek100 \
    --train_csv /kaggle/working/ek100-ann/EPIC_100_train.csv \
    --val_csv /kaggle/working/ek100-ann/EPIC_100_validation.csv \
    --video_info /kaggle/working/ek100-ann/EPIC_100_video_info.csv \
    --out_dir /kaggle/working/work | grep -E '"val_clips"|out_of_range_clips|warnings' -A1

## Tải + eval (chạy chặn đến khi xong hoặc hết ngân sách 10.5 giờ)
Log in mỗi khi tải xong một video: `[ok] P01_11 2.12 GB in 95s (…)`. Log eval của từng GPU nằm ở `logits_orig/eval_gpu*.log`.
Nếu tải chậm, tăng `--parallel_downloads` (số video tải cùng lúc) hoặc `--connections`.

In [ ]:
!python $R/scripts/stream_eval_ek100.py --work_dir /kaggle/working/work \
    --video_info /kaggle/working/ek100-ann/EPIC_100_video_info.csv --out_dir $OUT \
    --vitl_ckpt /tmp/ckpt/vitl.pt --probe_ckpt /tmp/ckpt/ek100-vitl-256.pt \
    --gpus 0 1 --anchors official action_start --dtype fp16 \
    --parallel_downloads 3 --connections 8 --max_videos_on_disk 10 --max_hours 10.5
!tail -n 3 $OUT/eval_gpu0.log $OUT/eval_gpu1.log

In [ ]:
# Results (if some videos are still missing, the numbers are marked PARTIAL: run the notebook again)
import subprocess
for anchor in ("official", "action_start"):
    cmd = (f"python {R}/scripts/compute_metrics.py --work_dir /kaggle/working/work --logits_dir {OUT}/{anchor} "
           f"--out /kaggle/working/verify_orig_{anchor}.json")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"[{anchor}] PARTIAL:", r.stderr.strip() or r.stdout.strip())
        r = subprocess.run(cmd + " --allow_partial", shell=True, capture_output=True, text=True)
    print(f"== {anchor} ==\n" + r.stdout)

In [ ]:
# Diagnostics on the official-protocol logits (counting schemes, bootstrap, ...), CPU only
!python $R/scripts/diagnose_ek100.py --work_dir /kaggle/working/work --logits_dir $OUT/official \
    --bootstrap 100 --out /kaggle/working/diagnose_orig.json | grep -E '"clean"|official_64gpu"|lots_of_15|bootstrap_std|world_size_range' -A2

### Đọc kết quả
- `official`: đếm như lần chạy 64 GPU của Meta, dùng để so với **32.7**.
- `clean`: mỗi clip đếm đúng 1 lần.
- `lots_of_15_1gpu_bs4` (trong phần diagnostics): đếm như notebook cũ chạy theo lô 15 video (đã báo cáo 31.24).
- `action_start`: giao thức anticipation chuẩn của EK100 (clip dừng 1s trước khi action bắt đầu).

Gửi lại output của 2 cell cuối.